# F6 Top-Field Negative-Flux Investigation

This notebook isolates the top-center F6 cluster, compares observed/no-DIG and DIG-subtracted line fluxes, highlights the selected regions on the field map, and plots them on BPT diagrams.

The selection box and line list are defined near the top so they can be adjusted without touching the plotting code.

In [ ]:
from pathlib import Path
import sys

_here = Path.cwd().resolve()
_candidates = (_here, *_here.parents)
REPO_ROOT = next((candidate for candidate in _candidates if (candidate / "m33_pipeline").is_dir()), None)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate repo root from {Path.cwd()}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from m33_pipeline.notebook_setup import prepare_notebook

REPO_ROOT = prepare_notebook(REPO_ROOT)
print(f"Notebook working directory set to: {REPO_ROOT}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
from astropy.io import fits
from astropy.wcs import WCS
from IPython.display import display

plt.rcParams.update({
    "font.family": "serif",
    "mathtext.fontset": "cm",
    "axes.linewidth": 1.4,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "text.usetex": False,
})

## Configuration

In [ ]:
FIELD = "F6"
CATALOG_PATH = Path("CATALOGS/flux_catalogs/summed_map/dig_subtracted/flux_catalog_F6.csv")
NO_DIG_CATALOG_PATH = Path("CATALOGS/flux_catalogs/summed_map/no_dig/flux_catalog_F6.csv")
IMAGE_PATH = Path("../M33-Maps/M33-F6/M33-F6_SN3.LineMaps.map.Ha+OIII.1x1.amplitude.fits")
WCS_PATH = Path("../M33-Maps/M33-F6/M33F6-Haflux.fits")
BOUNDARY_PATH = Path("Boundary_maps/Boundary_map_100pc/Boundary_map_F6.fits")

OUTDIR = Path("PAPER_PLOTS/summed_map/dig_subtracted/F6_investigation")
OUTDIR.mkdir(parents=True, exist_ok=True)

# Tune this box after looking at the first map. Pixel coordinates use the F6 map frame.
SUSPECT_X_RANGE = (650, 1300)
SUSPECT_Y_RANGE = (0, 360)

# BPT lines plus the second SII line for checking negative fluxes.
CHECK_LINES = ["Halpha", "Hbeta", "[OIII]5007", "[NII]6583", "[SII]6716", "[SII]6731"]
BPT_CLASS_COL = "BPT_class_sum_dered"

print(f"Figures will be saved to: {OUTDIR}")

## Load Catalogs And Define Suspect Regions

In [ ]:
cat = pd.read_csv(CATALOG_PATH)
cat_nodig = pd.read_csv(NO_DIG_CATALOG_PATH) if NO_DIG_CATALOG_PATH.exists() else None

if "field" in cat.columns:
    cat = cat.loc[cat["field"].astype(str).eq(FIELD)].copy()
if cat_nodig is not None and "field" in cat_nodig.columns:
    cat_nodig = cat_nodig.loc[cat_nodig["field"].astype(str).eq(FIELD)].copy()

if "primary" in cat.columns:
    cat = cat.loc[cat["primary"].fillna(True)].copy()
if cat_nodig is not None and "primary" in cat_nodig.columns:
    cat_nodig = cat_nodig.loc[cat_nodig["primary"].fillna(True)].copy()

cat = cat.reset_index(drop=True)
print(f"Loaded {len(cat)} primary F6 regions from {CATALOG_PATH}")
print(f"No-DIG comparison catalog: {NO_DIG_CATALOG_PATH if cat_nodig is not None else 'not found; using *_sum_nodig columns in the DIG-subtracted catalog'}")

In [ ]:
def flux_col(line, mode):
    return f"F_{line}_sum_{mode}"


def snr_col(line, mode):
    return f"SNR_{line}_sum_{mode}"


def numeric_col(df, col):
    if col not in df.columns:
        return pd.Series(np.nan, index=df.index, dtype=float)
    return pd.to_numeric(df[col], errors="coerce")


def any_negative_flux(df, mode, lines=CHECK_LINES):
    mask = pd.Series(False, index=df.index)
    for line in lines:
        col = flux_col(line, mode)
        if col in df.columns:
            mask |= numeric_col(df, col).lt(0)
    return mask


def negative_line_names(row, mode, lines=CHECK_LINES):
    neg = []
    for line in lines:
        col = flux_col(line, mode)
        if col in row.index and pd.notna(row[col]) and row[col] < 0:
            neg.append(line)
    return ", ".join(neg) if neg else ""


def clean_bpt_class(value):
    if pd.isna(value):
        return "Unclassified"
    value = str(value).strip()
    return value if value else "Unclassified"

in_top_box = (
    numeric_col(cat, "x").between(*SUSPECT_X_RANGE)
    & numeric_col(cat, "y").between(*SUSPECT_Y_RANGE)
)
pre_negative = any_negative_flux(cat, "nodig")
post_negative = any_negative_flux(cat, "digsub")
non_sf = ~cat[BPT_CLASS_COL].map(clean_bpt_class).eq("Star-forming") if BPT_CLASS_COL in cat.columns else pd.Series(True, index=cat.index)

# Main sample: top-center regions that are either non-SF or have any negative checked line.
cat["F6_top_box"] = in_top_box
cat["negative_pre_dig"] = pre_negative
cat["negative_post_dig"] = post_negative
cat["negative_either"] = pre_negative | post_negative
cat["suspect_top_cluster"] = in_top_box & (non_sf | cat["negative_either"])
cat["negative_lines_pre_dig"] = cat.apply(lambda row: negative_line_names(row, "nodig"), axis=1)
cat["negative_lines_post_dig"] = cat.apply(lambda row: negative_line_names(row, "digsub"), axis=1)

suspects = cat.loc[cat["suspect_top_cluster"]].copy()
print(f"Top box regions: {int(in_top_box.sum())}")
print(f"Top box + non-SF/negative selected regions: {len(suspects)}")
print(f"Selected with negative observed/no-DIG flux: {int(suspects['negative_pre_dig'].sum())}")
print(f"Selected with negative DIG-subtracted flux: {int(suspects['negative_post_dig'].sum())}")

cols = ["region_id", "x", "y", BPT_CLASS_COL, "negative_lines_pre_dig", "negative_lines_post_dig"]
with pd.option_context("display.max_rows", 200, "display.max_colwidth", 80):
    display(suspects[cols].sort_values(["y", "x"]))

In [ ]:
# Detailed flux/SNR table for the selected regions.
summary_cols = ["region_id", "x", "y", BPT_CLASS_COL, "negative_lines_pre_dig", "negative_lines_post_dig"]
for line in CHECK_LINES:
    for mode in ["nodig", "digsub"]:
        for prefix in ["F", "SNR"]:
            col = f"{prefix}_{line}_sum_{mode}"
            if col in cat.columns:
                summary_cols.append(col)

suspect_summary = suspects[summary_cols].sort_values(["y", "x"]).copy()
summary_path = OUTDIR / "F6_top_cluster_negative_flux_summary.csv"
suspect_summary.to_csv(summary_path, index=False)
print(f"Saved summary table: {summary_path}")
with pd.option_context("display.max_rows", 200, "display.max_columns", 80, "display.width", 220):
    display(suspect_summary)

## Map Highlight

In [ ]:
def load_image(path):
    data = np.squeeze(fits.getdata(path)).astype(float)
    data = np.where(np.isfinite(data), data, np.nan)
    with np.errstate(divide="ignore", invalid="ignore"):
        log_data = np.log10(data)
    return np.where(np.isfinite(log_data), log_data, np.nan)


def label_edges(label_map):
    lab = np.asarray(label_map)
    valid = np.isfinite(lab) & (lab > 0)
    edges = np.zeros(lab.shape, dtype=bool)

    v1 = valid[:, :-1]
    v2 = valid[:, 1:]
    l1 = lab[:, :-1]
    l2 = lab[:, 1:]
    boundary_x = (v1 != v2) | (v1 & v2 & (l1 != l2))
    edges[:, :-1] |= boundary_x
    edges[:, 1:] |= boundary_x

    v1 = valid[:-1, :]
    v2 = valid[1:, :]
    l1 = lab[:-1, :]
    l2 = lab[1:, :]
    boundary_y = (v1 != v2) | (v1 & v2 & (l1 != l2))
    edges[:-1, :] |= boundary_y
    edges[1:, :] |= boundary_y
    return edges


def rgba_overlay(mask, color, alpha=1.0):
    rgba = np.zeros(mask.shape + (4,), dtype=float)
    rgba[..., :3] = mcolors.to_rgb(color)
    rgba[..., 3] = mask.astype(float) * alpha
    return rgba


def ids_to_edge_mask(label_map, ids):
    ids = np.asarray(list(ids), dtype=int)
    if len(ids) == 0:
        return np.zeros(label_map.shape, dtype=bool)
    region_mask = np.isin(label_map, ids)
    return label_edges(np.where(region_mask, label_map, 0))

image = load_image(IMAGE_PATH)
boundary_map = np.squeeze(fits.getdata(BOUNDARY_PATH)).astype(float)

all_edges = label_edges(boundary_map)
suspect_edges = ids_to_edge_mask(boundary_map, suspects["region_id"].dropna().astype(int))
negative_post_edges = ids_to_edge_mask(boundary_map, suspects.loc[suspects["negative_post_dig"], "region_id"].dropna().astype(int))

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(image, origin="lower", cmap="rainbow", vmin=-18, vmax=-15.0)
ax.imshow(rgba_overlay(all_edges, "0.1", alpha=0.35), origin="lower", interpolation="nearest")
ax.imshow(rgba_overlay(suspect_edges, "deepskyblue", alpha=0.95), origin="lower", interpolation="nearest")
ax.imshow(rgba_overlay(negative_post_edges, "crimson", alpha=1.0), origin="lower", interpolation="nearest")

ax.scatter(suspects["x"], suspects["y"], s=24, facecolors="none", edgecolors="white", linewidths=1.2, zorder=5)
for _, row in suspects.iterrows():
    ax.text(row["x"] + 8, row["y"] + 8, str(int(row["region_id"])), color="white", fontsize=8, zorder=6)

ax.axvspan(SUSPECT_X_RANGE[0], SUSPECT_X_RANGE[1], ymin=0, ymax=1, color="white", alpha=0.04)
ax.set_xlim(550, 1400)
ax.set_ylim(0, 430)
ax.set_xlabel("x [pix]")
ax.set_ylabel("y [pix]")
ax.set_title(f"F6 top-center suspect regions: N={len(suspects)}; red = negative after DIG")
cb = fig.colorbar(im, ax=ax, pad=0.02)
cb.set_label("log10(Halpha + [O III])")
fig.tight_layout()
fig.savefig(OUTDIR / "F6_top_cluster_map_highlight.png", dpi=250, bbox_inches="tight")
plt.show()

## BPT Comparison: Observed/No-DIG vs DIG-Subtracted

In [ ]:
def kauffmann03(x):
    return 0.61 / (x - 0.05) + 1.3


def kewley01(x):
    return 0.61 / (x - 0.47) + 1.19


def bpt_xy(df, mode):
    ha = numeric_col(df, flux_col("Halpha", mode))
    hb = numeric_col(df, flux_col("Hbeta", mode))
    oiii = numeric_col(df, flux_col("[OIII]5007", mode))
    nii = numeric_col(df, flux_col("[NII]6583", mode))
    good = (ha > 0) & (hb > 0) & (oiii > 0) & (nii > 0)
    x = pd.Series(np.nan, index=df.index, dtype=float)
    y = pd.Series(np.nan, index=df.index, dtype=float)
    x.loc[good] = np.log10(nii.loc[good] / ha.loc[good])
    y.loc[good] = np.log10(oiii.loc[good] / hb.loc[good])
    return x, y, good


def plot_bpt_mode(ax, df, mode, title):
    x_all, y_all, good_all = bpt_xy(df, mode)
    is_suspect = df["suspect_top_cluster"].fillna(False)
    class_values = df[BPT_CLASS_COL].map(clean_bpt_class) if BPT_CLASS_COL in df.columns else pd.Series("Unclassified", index=df.index)
    is_sf = class_values.eq("Star-forming")

    ax.scatter(x_all[good_all & ~is_suspect & ~is_sf], y_all[good_all & ~is_suspect & ~is_sf], s=10, c="0.70", alpha=0.6, label="Other non-SF")
    ax.scatter(x_all[good_all & ~is_suspect & is_sf], y_all[good_all & ~is_suspect & is_sf], s=10, c="black", alpha=0.55, label="Other BPT SF")
    ax.scatter(x_all[good_all & is_suspect], y_all[good_all & is_suspect], s=60, c="deepskyblue", edgecolors="black", linewidths=0.8, label="Top F6 sample", zorder=5)

    bad_suspects = df.loc[is_suspect & ~good_all]
    if len(bad_suspects):
        ax.scatter([], [], marker="x", c="crimson", s=60, label=f"Top F6 sample off-BPT (N={len(bad_suspects)})")

    for _, row in df.loc[is_suspect & good_all].iterrows():
        ax.text(x_all.loc[row.name] + 0.015, y_all.loc[row.name] + 0.015, str(int(row["region_id"])), fontsize=7, color="black")

    xx1 = np.linspace(-1.5, 0.03, 300)
    xx2 = np.linspace(-1.5, 0.45, 300)
    ax.plot(xx1, kauffmann03(xx1), color="black", lw=1.2, ls="--", label="Kauffmann+03")
    ax.plot(xx2, kewley01(xx2), color="black", lw=1.2, ls=":", label="Kewley+01")
    ax.set_xlim(-1.6, 0.7)
    ax.set_ylim(-1.2, 1.4)
    ax.set_xlabel("log([N II] 6583 / Halpha)")
    ax.set_ylabel("log([O III] 5007 / Hbeta)")
    ax.set_title(title)
    ax.legend(frameon=False, fontsize=9, loc="lower left")
    return bad_suspects

fig, axes = plt.subplots(1, 2, figsize=(13, 5.8), sharex=True, sharey=True)
bad_pre = plot_bpt_mode(axes[0], cat, "nodig", "Observed/no-DIG fluxes")
bad_post = plot_bpt_mode(axes[1], cat, "digsub", "DIG-subtracted fluxes")
fig.tight_layout()
fig.savefig(OUTDIR / "F6_top_cluster_BPT_pre_post_DIG.png", dpi=250, bbox_inches="tight")
plt.show()

print("Selected regions that cannot be placed on BPT because one or more required BPT lines are non-positive:")
for label, bad in [("observed/no-DIG", bad_pre), ("DIG-subtracted", bad_post)]:
    print(f"\n{label}: N={len(bad)}")
    if len(bad):
        display(bad[["region_id", "x", "y", BPT_CLASS_COL, "negative_lines_pre_dig", "negative_lines_post_dig"]].sort_values(["y", "x"]))

## Pre/Post DIG Flux Sign Summary

In [ ]:
rows = []
for _, row in suspects.sort_values(["y", "x"]).iterrows():
    for line in CHECK_LINES:
        pre = row.get(flux_col(line, "nodig"), np.nan)
        post = row.get(flux_col(line, "digsub"), np.nan)
        rows.append({
            "region_id": int(row["region_id"]),
            "x": row["x"],
            "y": row["y"],
            "line": line,
            "F_observed_nodig": pre,
            "F_digsub": post,
            "negative_observed_nodig": pd.notna(pre) and pre < 0,
            "negative_digsub": pd.notna(post) and post < 0,
            "changed_positive_to_negative": (pd.notna(pre) and pd.notna(post) and pre > 0 and post < 0),
        })

sign_table = pd.DataFrame(rows)
sign_path = OUTDIR / "F6_top_cluster_pre_post_DIG_flux_signs.csv"
sign_table.to_csv(sign_path, index=False)
print(f"Saved sign table: {sign_path}")

pivot = sign_table.pivot_table(
    index=["region_id", "x", "y"],
    columns="line",
    values=["negative_observed_nodig", "negative_digsub", "changed_positive_to_negative"],
    aggfunc="first",
)
with pd.option_context("display.max_rows", 200, "display.max_columns", 80, "display.width", 220):
    display(pivot)

problem_lines = (
    sign_table.loc[sign_table["negative_observed_nodig"] | sign_table["negative_digsub"]]
    .groupby("line")[["negative_observed_nodig", "negative_digsub", "changed_positive_to_negative"]]
    .sum()
    .astype(int)
    .sort_values("negative_digsub", ascending=False)
)
print("Negative-line counts within selected top F6 sample:")
display(problem_lines)

## Region IDs For Quick Follow-Up

In [ ]:
print("Selected top F6 region IDs:")
print(", ".join(str(int(rid)) for rid in suspects.sort_values(["y", "x"])["region_id"]))

print()
print("Selected region IDs with negative observed/no-DIG flux:")
print(", ".join(str(int(rid)) for rid in suspects.loc[suspects["negative_pre_dig"]].sort_values(["y", "x"])["region_id"]))

print()
print("Selected region IDs with negative DIG-subtracted flux:")
print(", ".join(str(int(rid)) for rid in suspects.loc[suspects["negative_post_dig"]].sort_values(["y", "x"])["region_id"]))
